### 0. Environments

In [5]:
!pip install -q ir_datasets bm25s sentence-transformers faiss-gpu-cu11 ranx tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 23.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 44.8 MB/s eta 0:00:00


In [8]:
import ir_datasets
from tqdm import tqdm
import ir_datasets
from itertools import islice
import bm25s
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [9]:
# configurations
MAX_QUERIES = 200
MAX_DOCS = 20_000
TOP_K = 100
BATCH_SIZE = 128
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

### 1. Download MSMARCO dataset

In [10]:
!cp -r /kaggle/input/datasets/hongkimgip/msmarco-passage/msmarco-passage /root/.ir_datasets/msmarco-passage

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [11]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, queries, qrels = download_and_preview_msmarco()


========== SAMPLE PASSAGES ==========

--- Sample Passage 1 ---
doc_id: 0
text: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

========== SAMPLE QUERIES ==========

--- Sample Querie 1 ---
query_id: 1048585
text: what is paula deen's brother

========== SAMPLE QRELS ==========

--- Sample Qrel 1 ---
query_id: 300674
doc_id: 7067032
relevance: 1

========== LOADING FULL CORPUS ==========


Loading passages: 8841823it [00:37, 238302.43it/s]


Total passages loaded: 8,841,823

========== LOADING QUERIES ==========


Loading queries: 6980it [00:00, 440960.39it/s]


Total queries loaded: 6,980

========== LOADING QRELS ==========


Loading qrels: 7437it [00:00, 388567.57it/s]

Total qrels queries loaded: 6,980


### 2. Retrieve

#### Sparse:

In [12]:
class BM25SRetriever:
    def __init__(self, corpus):
        self.doc_ids = list(corpus.keys())
        self.doc_texts = [corpus[doc_id] for doc_id in corpus.keys()]
        self.retriever = bm25s.BM25()

    def build(self, show_progress=True):
        """
        Build BM25 index từ ir_datasets corpus.
        """

        corpus_tokens = bm25s.tokenize(self.doc_texts)
        self.retriever.index(corpus_tokens)

        print(f"Indexed {len(self.doc_texts):,} documents")

    def search(self, queries, top_k=100):
        """
        return:
            {
                doc_id: score,
                ...
            }
        """

        if isinstance(queries, str):
            queries = [queries]

        query_tokens = bm25s.tokenize(queries)

        results, scores = self.retriever.retrieve(
            query_tokens,
            corpus=self.doc_ids,
            k=top_k,
        )

        all_results = []

        for doc_id_list, score_list in zip(results, scores):
            run = {
                str(doc_id): float(score)
                for doc_id, score in zip(doc_id_list, score_list)
            }
            all_results.append(run)

        return all_results

Build index:

In [45]:
bm25_retriever = BM25SRetriever(corpus)
bm25_retriever.build()

Split strings:   0%|          | 0/8841823 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/8841823 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/8841823 [00:00<?, ?it/s]

Indexed 8,841,823 documents


Search thử:

In [56]:
query_ids = list(queries.keys())[:5]
query_texts = [queries[qid] for qid in query_ids]
bm25_results = bm25_retriever.search(query_texts, top_k=10)

for qid, qtext, result in zip(query_ids, query_texts, bm25_results):
    print("=" * 80)
    print("QID:", qid)
    print("Query:", qtext)
    for doc_id, score in list(result.items())[:3]:
        print(score, '|', doc_id, '|', corpus[doc_id][:200])

Split strings:   0%|          | 0/5 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/5 [00:00<?, ?it/s]

QID: 1048585
Query: what is paula deen's brother
16.036605834960938 | 7187158 | Paula Deen and her brother Earl W. Bubba Hiers are being sued by a former general manager at Uncle Bubba's… Paula Deen and her brother Earl W. Bubba Hiers are being sued by a former general manager at
15.201703071594238 | 7187157 | The New York Times. U.S. | National Briefing | South. Georgia: Paula Deen and Brother Shut a Restaurant. The celebrity chef Paula Deen and her younger brother, Bubba Hiers, have closed a Savannah seaf
15.172785758972168 | 3856133 | Viewing Tweets won't unblock @Paula_Deen. 1  Paula Deen‏Verified account @Paula_Deen 8h8 hours ago. 2  Paula Deen‏Verified account @Paula_Deen 23h23 hours ago. 3  Paula Deen‏Verified account @Paula_De
QID: 2
Query:  Androgen receptor define
12.475543975830078 | 1782337 | Enzalutamide is an androgen receptor inhibitor that acts on different steps in the androgen receptor signaling pathway. Enzalutamide has been shown to competitively inhibit androgen bi

#### Dense:

In [13]:
class DenseFaissRetriever:
    def __init__(
        self,
        corpus,
        model_name="BAAI/bge-small-en-v1.5",
        batch_size=128,
        device=None,
    ):
        self.doc_ids = list(corpus.keys())
        self.doc_texts = [corpus[doc_id] for doc_id in corpus.keys()]
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = device

        self.model = SentenceTransformer(model_name, device=device)
        self.index = None
        self.embeddings = None

    def build(self):
        embeddings = self.model.encode(
            self.doc_texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        embeddings = embeddings.astype("float32")

        dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(embeddings)

        self.embeddings = embeddings
        self.index = index

        print("FAISS index size:", self.index.ntotal)
        print("Embedding dim:", dim)

    def search(self, queries, top_k=100):
        query_embeddings = self.model.encode(
            queries,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        scores, indices = self.index.search(query_embeddings, top_k)

        all_results = []

        for score_list, idx_list in zip(scores, indices):
            run = {}
            for score, idx in zip(score_list, idx_list):
                if idx == -1:
                    continue
                doc_id = self.doc_ids[idx]
                run[str(doc_id)] = float(score)
            all_results.append(run)

        return all_results

Build index:

In [ ]:
dense_retriever = DenseFaissRetriever(
    corpus=corpus,
    model_name=EMBEDDING_MODEL,
    batch_size=128,
)

dense_retriever.build()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/69077 [00:00<?, ?it/s]

Test:

In [ ]:
query_ids = list(queries.keys())[:5]
query_texts = [queries[qid] for qid in query_ids]
dense_results = dense_retriever.search(query_texts, top_k=10)

for qid, qtext, result in zip(query_ids, query_texts, dense_results):
    print("=" * 80)
    print("QID:", qid)
    print("Query:", qtext)
    for doc_id, score in list(result.items())[:3]:
        print(score, doc_id, corpus[doc_id][:200])

### 3. Fusion

In [ ]:
def rrf_fusion_one(sparse_run, dense_run, k=60, top_k=100):
    fused = {}

    for rank, doc_id in enumerate(sparse_run.keys(), start=1):
        fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)

    for rank, doc_id in enumerate(dense_run.keys(), start=1):
        fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)

    fused = dict(
        sorted(
            fused.items(),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]
    )

    return fused


def rrf_fusion_all(sparse_results, dense_results, k=60, top_k=100):
    fused_results = []

    for sparse_run, dense_run in zip(sparse_results, dense_results):
        fused_results.append(
            rrf_fusion_one(
                sparse_run=sparse_run,
                dense_run=dense_run,
                k=k,
                top_k=top_k,
            )
        )

    return fused_results

#### Full retrieval:

In [ ]:
TOP_K = 100

bm25_results = bm25_retriever.search(query_texts, top_k=TOP_K)
dense_results = dense_retriever.search(query_texts, top_k=TOP_K)

hybrid_results = rrf_fusion_all(
    sparse_results=bm25_results,
    dense_results=dense_results,
    k=60,
    top_k=TOP_K,
)

### 4. Evaluate

In [ ]:
from ranx import Qrels, Run, evaluate

def make_run_dict(query_ids, results):
    run_dict = {}

    for qid, result in zip(query_ids, results):
        run_dict[str(qid)] = {
            str(doc_id): float(score)
            for doc_id, score in result.items()
        }

    return run_dict


qrels = Qrels({
    str(qid): {
        str(doc_id): int(rel)
        for doc_id, rel in rel_docs.items()
    }
    for qid, rel_docs in qrels_dict.items()
})

bm25_run = Run(
    make_run_dict(query_ids, bm25_results),
    name="bm25s",
)

dense_run = Run(
    make_run_dict(query_ids, dense_results),
    name="bge_faiss",
)

hybrid_run = Run(
    make_run_dict(query_ids, hybrid_results),
    name="bm25s_bge_rrf",
)

In [ ]:
metrics = [
    "mrr@10",
    "ndcg@10",
    "recall@10",
    "recall@100",
    "map@100",
]

bm25_scores = evaluate(qrels, bm25_run, metrics)
dense_scores = evaluate(qrels, dense_run, metrics)
hybrid_scores = evaluate(qrels, hybrid_run, metrics)

print("BM25S:", bm25_scores)
print("Dense:", dense_scores)
print("Hybrid RRF:", hybrid_scores)